# Lab 2 第一支生成程式：把雲端 LLM 叫起來（教學版）

**今天的目標：** 寫出第一支**會呼叫 LLM、印出它生成文字**的程式。Lab 1 的分數是你自己算的；這次你的 code 只負責「把問題送出去、把生成的回答收回來」。

**這本是教學版：** 核心程式碼挖成 `____`，跟著 `# TODO` 提示自己填；每個小作業下面有摺疊的參考解，做完再打開對答案。

> 生成跑在**雲端**，你的筆電不需要 GPU。財務數字全為**虛構教學範例**（統一用虛構公司「宏圖飲料」），非真實行情。

In [ ]:
# 📦 先跑這一格：一次裝齊今天要用的套件（已裝好的會直接跳過；裝不起來看 README）
!pip install -q langchain-ollama langchain-google-genai langchain-community langchain-text-splitters faiss-cpu requests

## 🔧 第 0 步：環境自我檢查（四格，跑一格看一個燈）

這四格**不用填，直接跑**。安裝步驟在 `README.md`——紅燈時照它指的那一步回頭補。

> 燈號怎麼看：**綠燈 ✅ 就往下**。第 2、3 格紅燈 → 今天做不下去，一定要先修好；第 1、4 格紅燈 → **今天照樣能做完**（生成走雲端），但**下午的 Lab 3 會卡住**，趁現在補。

In [ ]:
# ✅ 檢查 1：本機 Ollama 服務通不通（Lab 3 的 embedding 要用它；今天的生成走雲端，不靠它）
import requests

try:
    r = requests.get("http://localhost:11434/api/tags", timeout=3)
    print("✅ 本機 Ollama 服務通了")
except Exception:
    print("❌ 連不上本機 Ollama（http://localhost:11434）")
    print("   → 開一次 Ollama App，或在終端機下 ollama serve；沒裝的話看 README 第 1 步")
    print("   → 今天的 Lab 2 走雲端，這個紅燈不影響；但 Lab 3 一定要修好")

In [ ]:
# ✅ 檢查 2：設定 API key（今天呼叫雲端的「門票」，沒有它整個 Lab 做不下去）
import os

# 👇 同學：把引號中間換成老師給你的 OLLAMA key（整段貼進去，前後別留空白）
os.environ["OLLAMA_API_KEY"] = "在這裡貼上你的 OLLAMA key"

key = os.environ.get("OLLAMA_API_KEY")
if key and key != "在這裡貼上你的 OLLAMA key":
    print("✅ OLLAMA_API_KEY 已設定（長度", len(key), "個字元）")
else:
    print("❌ 還沒貼 key → 把上面那行引號中間換成你的 OLLAMA key，再重跑這一格")

# 🔒 只印長度、不印 key 本身——key 等於你帳號的鑰匙。
#    貼了 key 的 notebook 別上傳 GitHub、別傳給別人、別截圖給人看。

In [ ]:
# ✅ 檢查 3：套件裝了沒（langchain-ollama 是今天要用的；另外兩個是 Lab 3 的預備）
try:
    from langchain_ollama import ChatOllama
    print("✅ langchain-ollama（今天要用）")
except Exception:
    print("❌ langchain-ollama 沒裝 → 看 README 第 4 步")

try:
    import faiss
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    print("✅ faiss-cpu + langchain-text-splitters（Lab 3 要用）")
except Exception:
    print("⚠️ Lab 3 的套件還缺 → 今天不影響，下午前補裝（README 第 4 步）")

In [ ]:
# ✅ 檢查 4：Lab 3 的 embedding 模型拉了沒（bge-m3，1.2GB）
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=3)
    names = []
    for m in r.json()["models"]:
        names.append(m["name"])

    if "bge-m3:latest" in names:
        print("✅ bge-m3 已備妥（Lab 3 用得到）")
    else:
        print("⚠️ 還沒拉 bge-m3 → 終端機下：ollama pull bge-m3")
    print("   本機現有模型：", names)
except Exception:
    print("⚠️ 本機 Ollama 沒通，無法確認（同檢查 1，Lab 3 前要修好）")


---
## 🟦 練 A：把雲端 LLM 叫起來，印出它生成的第一句話

**雲端生成只有三件事**：指定**哪個模型**、告訴它**打去哪**（雲端不是本機）、附上**門票**（你的 API key）。設好之後，一行 `invoke` 就能問問題。

### A1・接上雲端生成模型

下面 code 只是「建立連線物件」，**還沒真的問問題**，所以很快、不扣額度。

**預期輸出：** 印出模型名 `gemma4:cloud`、打去 `https://ollama.com`、門票已帶上。

In [ ]:
import os
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gemma4:cloud",          # 雲端模型要帶 cloud tag；漏了它會去找「本機」的同名模型 → model not found
    base_url="https://ollama.com", # 打到雲端（不寫這行就是打你自己的電腦 localhost）
    client_kwargs={"headers": {"Authorization": f"Bearer {os.environ['OLLAMA_API_KEY']}"}},
    # 💡 os.environ 是「環境字典」——開頭「檢查 2」那格已經把 key 放進去，這裡直接拿來用
    # 💡 Bearer <key> 是把門票夾在請求上的標準寫法，照樣板用即可
)

print("模型：", llm.model)
print("打去：", llm.base_url)
print("門票：已夾在請求上（key 本身不印出來）")

### A2・先看它「原本」長什麼樣（這一格會真的打雲端）

先不要急著取文字——**先看雲端送回來的原始包裹長怎樣**。最後一行故意不寫 `print`，讓 Jupyter 把物件整個攤開給你看。

**預期輸出：** 一個 `AIMessage(...)` 物件。你要的那句中文**就在裡面**，但外面還裹著一大堆標籤資料——**先看清楚它的原始長相**，下一格再把文字挑出來。

> ⏱ 這格要等**幾秒**（雲端在生成），**不是當掉**。

In [ ]:
resp = llm.____("用一句話解釋什麼是毛利率")
# TODO：把問題「送出去」給雲端模型、等它生成完再回來——ChatOllama 送出問題的那個動作叫什麼？
#       （這一行才是真正打雲端、會扣額度的一行）

resp   # 不寫 print，直接讓 Jupyter 攤開整個回傳物件

### A3・從包裹裡把「文字」挑出來

上一格看到了：回傳的不是一段字，是一個**物件**。要印給人看，得先把純文字那一格取出來。

**預期輸出：** 乾乾淨淨一句中文（沒有 `AIMessage(...)`、沒有一堆 metadata）。

In [ ]:
answer = resp.____
# TODO：從上一格那個回傳物件裡，把「純文字回答」那一格取出來
#       （欄位名就寫在上一格的輸出裡，自己找）

print(answer)

### A4・順便看一眼：這次用掉多少 token？（不用填，直接跑）

LLM 不是一個字一個字生成，而是一個 **token（詞片段）** 一個一個生成——**雲端按 token 計費，你送出去的問題也算錢**，所以 token 數就是成本。

**預期輸出：** 像 `{'input_tokens': 21, 'output_tokens': 28, 'total_tokens': 49}`（每次跑數字會略有不同）。

> 盯著看最後兩行：**token 不等於字**。回答的字數會比 token 數還多——**一個 token 常常吃掉一個以上的中文字**，所以不能用「幾個字」去推「要花多少錢」。

In [ ]:
print("token 用量：", resp.usage_metadata)
print()
print("你的問題（input）＝", resp.usage_metadata["input_tokens"], "個 token")
print("它的回答（output）＝", resp.usage_metadata["output_tokens"], "個 token")
print("但這句回答其實有", len(answer), "個字 ← 字數跟 token 數不一樣，別把兩者當同一件事")

### 📝 小作業 A

1. 換一個**你自己的金融問題**問它（例如「用兩句話解釋什麼是本益比」），印出回答。
2. 再看一次 `usage_metadata`：問題變長，`input_tokens` 有跟著變大嗎？要它答長一點，`output_tokens` 呢？
3. **⭐ 進階（選做）：** 同一個問題連問兩次，兩次的回答**一模一樣嗎**？（先猜再跑。）

<details><summary>📖 做完再看：參考解（參考解不只一種，思路對就好）</summary>

```python
resp2 = llm.invoke("用兩句話解釋什麼是本益比")
print(resp2.content)
print("token 用量：", resp2.usage_metadata)

# ⭐ 進階：同題連問兩次
a = llm.invoke("用一句話解釋什麼是本益比").content
b = llm.invoke("用一句話解釋什麼是本益比").content
print("第一次：", a)
print("第二次：", b)
```

**結論：** 問題越長 `input_tokens` 越大、答案越長 `output_tokens` 越大——**兩邊都算錢**，所以實務上「prompt 不要無謂地長」。
同題連問兩次通常**不會一模一樣**：LLM 每一步都是在「挑下一個 token」，這個挑選帶有隨機性——所以它不是查表，是**每次重新生成一遍**。
</details>

In [ ]:
# 📝 小作業 A：在這格作答
# Q1 + Q2：換一個你自己的金融問題問它（例：用兩句話解釋什麼是本益比），印出回答；
#          再看一次 usage_metadata——問題變長 input_tokens 有變大嗎？要它答長一點 output_tokens 呢？
# ⭐ 進階（選做）：同一個問題連問兩次，兩次回答一模一樣嗎？（先猜再跑）

# 你的答案：

---
## 🟩 練 B：包成函式，然後轉一個旋鈕

每次都寫 `llm.invoke(...).content` 太累——先包成一個函式，再來玩最常用的旋鈕：**`num_predict`（回答長度上限）**。

### B1・包成一個小函式

把「送問題 → 取文字」縮成一個 `ask_llm()`，之後想問什麼就一行。

**預期輸出：** 一句話的財報摘要。

In [ ]:
def ask_llm(question):
    return llm.____
    # TODO：把前面練過的兩個動作「串成一行」——先送出去，再把純文字取出來。
    #       ⚠️ 兩個坑：① 要送的是這個函式收到的 question，不是寫死的問句；
    #                  ② 少了取文字那一步，回傳的會是整包物件而不是一句話。

print(ask_llm("把這段財報摘要成一句話，只輸出那一句、不要提供多個版本：宏圖飲料 2026 Q1 營收 12.5 億元、毛利率 38%、EPS 2.1 元。"))

### B2・`num_predict`：控制「最多生幾個 token 就收手」

**預期輸出：** 一段**講到一半就斷掉**的回答（故意的）——它連第一句都還沒說完。

> 💡 設太小 → 被切斷；設大 → 完整但更慢、更花錢。實務上依你要的回答長度調。

In [ ]:
llm_short = ChatOllama(
    model="gemma4:cloud",
    base_url="https://ollama.com",
    client_kwargs={"headers": {"Authorization": f"Bearer {os.environ['OLLAMA_API_KEY']}"}},
    ____=20,   # TODO：填上「最多生成幾個 token 就收手」的那個參數名
               #       （提示：num_ 開頭，後面接「預測」的英文——它預測的就是下一個 token）
)

print(llm_short.invoke("詳細解釋什麼是 EPS（每股盈餘），並舉例說明怎麼算").content)

### 📝 小作業 B

1. 用 `ask_llm()` 問一個**跟金融無關**的問題（隨便什麼都行），確認函式真的可以重複用。
2. 把 B2 的 `num_predict` 從 30 慢慢調大（60、120…），**找出「剛好講得完」的那個值**。
3. **⭐ 進階（選做）：** 不用 `num_predict`，改成**在 prompt 裡直接要求**「用 20 字以內回答」。兩種做法都能讓答案變短——它們差在哪？（先想再跑。提示：一個是「講到一半剪掉」，另一個是「請它自己講短一點」。）

<details><summary>📖 做完再看：參考解（參考解不只一種，思路對就好）</summary>

```python
print(ask_llm("用一句話說明為什麼天空是藍的"))     # 函式跟題材無關，什麼都能問

llm_120 = ChatOllama(
    model="gemma4:cloud",
    base_url="https://ollama.com",
    client_kwargs={"headers": {"Authorization": f"Bearer {os.environ['OLLAMA_API_KEY']}"}},
    num_predict=120,
)
print(llm_120.invoke("詳細解釋什麼是 EPS（每股盈餘）").content)

# ⭐ 進階：改用 prompt 要求它講短
print(ask_llm("用 20 字以內解釋什麼是 EPS（每股盈餘）"))
```

**結論：**
- `ask_llm()` 裡沒有任何跟金融有關的東西——它只是「送問題、取文字」，**題材是你 prompt 的事**。
- `num_predict` 要調到「比它想講的長度大一點」才不會被剪。
- ⭐ 兩種做法**結果很不一樣**：`num_predict` 是**硬剪**——它照樣想講長篇，只是被你從中間切斷，所以會斷在半句話。prompt 要求「20 字以內」是**請它一開始就講短**，句子是完整的。
  **實務上兩個一起用**：prompt 講清楚你要多短（拿到完整的短答案），`num_predict` 當保險絲（萬一它不聽話，也不會燒掉你一大筆 token）。
</details>

In [ ]:
# 📝 小作業 B：在這格作答
# Q1：用 ask_llm() 問一個「跟金融無關」的問題，確認函式真的可以重複用
# Q2：把 B2 的 num_predict 慢慢調大（60、120…），找出「剛好講得完」的那個值
# ⭐ 進階（選做）：不用 num_predict，改成在 prompt 裡直接要求「用 20 字以內回答」——兩種做法差在哪？
#     提示：一個是「講到一半剪掉」，另一個是「請它自己講短一點」。

# 你的答案：

---
## 🟥 練 C：它會生成——但它說的是對的嗎？

現在你已經會叫 LLM 生成了。接下來這兩格是**今天最重要的畫面**，也是下午 Lab 3（RAG）整堂課要治的病。

### C1・故意問它一個「它不可能知道」的數字

「宏圖飲料」是**虛構公司**——模型訓練時**不可能看過**它的財報。這一格**問題由你自己寫**。

**預期輸出：** 兩種都可能，**都成立**——
- **情況一（誠實）：** 它說知識庫裡查無這家公司，答不出來。
- **情況二（幻覺）：** 它**自信地掰一個數字**出來。

盯著看它到底走哪一種；如果它有講出「你可以提供相關文件給我」之類的話，把那句記住——**下一格你就要做這件事**。

In [ ]:
question = "____"
# TODO：自己寫一個問題，問「宏圖飲料」這家虛構公司的某個**具體財務數字**
#       （例如最近一季的 EPS、營收、毛利率⋯⋯挑一個）。
#       ⚠️ 不要問「什麼是毛利率」這種它本來就答得出來的常識題——
#          那樣它會答得好好的，等一下的對照就演不出來了。

print(ask_llm(question))

### C2・**同一個問題**，這次把資料一起送過去

上一格它答不出來（或掰了）——因為它**沒有你的資料，只能憑記憶**。
這一格**問題一個字都不改**，只多做一件事：**把資料跟問題一起送**。

**預期輸出：** 它**照著你給的資料**精準答出來（問 EPS 的話就是 **2.1 元**）。

In [ ]:
data = "宏圖飲料 2026 Q1（最近一季）營收 12.5 億元、毛利率 38%、EPS 2.1 元。（【範例】虛構資料）"
# question 沿用上一格那個一模一樣的問題，不要改它——這樣才看得出「差別只在有沒有給資料」

prompt = f"""
請只根據以下資料回答問題，資料沒寫的就說不知道。
資料：{____}
問題：{____}
"""
# TODO：把上面那兩個變數嵌進這段話裡——先讓它看資料，再問問題。

print(prompt)
print("---------- 它的回答 ----------")
print(ask_llm(prompt))

### 🔑 收束：你剛剛手工做了一次 RAG 的最後一刀

| | 你送出去的東西 | 它的回答 |
|---|---|---|
| **C1** | 只有問題 | 答不出來，或**自信地掰**（＝**幻覺**） |
| **C2** | **資料 ＋ 問題** | 精準答對 2.1 元 |

**同一個模型、同一個問題，差別只在「你有沒有把資料一起送過去」。**

那問題來了 👉 **如果你有 1000 份財報，總不能全部貼進 prompt 吧？**（塞不下，而且每個 token 都要錢。）

> **所以要先「把最相關的那幾段挑出來」——挑的方法，就是你上午 Lab 1 做的檢索。**
> **Lab 1（檢索） ＋ Lab 2（生成） ＝ 下午的 Lab 3：RAG。** 今天下午我們就把這兩半縫起來。

## 🌐 D 段（加映）：商用 API vs 開源模型 —— 同一個問題，問兩邊

到目前為止，你呼叫的都是**開源模型**（`gemma4`）。
下午區塊 3 會談另一條路：**商用託管 API**（Gemini / ChatGPT / Copilot）。

這一段讓你親眼看到兩條路的差別。**不用填空、直接跑就好**——重點是「知道怎麼做」。

> **這段可以整段跳過**，不影響 Lab 2 的驗收，也不影響下午的 Lab 3。

### D1・先拿一把 Gemini 的 key（跟 Ollama 那把是兩把不同的鑰匙）

到 **https://aistudio.google.com/apikey** 免費申請一把（Google 帳號登入即可），然後**把 key 直接貼進下面那格的引號中間**——跟開頭「檢查 2」貼 Ollama key 的做法一樣。

沒有 key 也沒關係——**留空字串不動就好**，下面每一格都有防呆，會直接告訴你「跳過」，不會噴錯。

In [ ]:
import os

# D 段要用 Gemini（商用 API 對照）。有 key 才跑，沒有可以整段跳過、不影響前面。
# 👇 有 Gemini key 的話，把引號中間換成你的 key；沒有就留「空字串」不動，D 段會自動跳過。
os.environ["GOOGLE_API_KEY"] = ""   # ← 貼你的 Gemini key（留空＝跳過 D 段）

GOOGLE_KEY = os.environ.get("GOOGLE_API_KEY", "")

if GOOGLE_KEY == "":
    print("⚠️ 還沒有 Gemini key —— D 段可以整段跳過，不影響其他部分")
    print("   想試的話：https://aistudio.google.com/apikey （免費）")
else:
    print("✅ 讀到 Gemini key，可以往下跑")

### D2・接上 Gemini —— 跟 A1 的 `ChatOllama` 對照著看

這是本段**最想讓你看到的一件事**：

```python
llm    = ChatOllama(model="gemma4:cloud", ...)              # A1：開源模型
gemini = ChatGoogleGenerativeAI(model="gemini-3.5-flash")   # D2：商用模型
```

**換一個 class，就換一家供應商——後面的 `.invoke()`、`.content`、`.usage_metadata` 通通一模一樣。**
這就是 LangChain 幫你抹平的東西：**換模型不用重寫程式**。

> 需要先裝套件：`pip install -U langchain-google-genai`（已寫進 requirements.txt）

In [ ]:
# 需要先裝：pip install -U langchain-google-genai
from langchain_google_genai import ChatGoogleGenerativeAI

gemini = None

if GOOGLE_KEY == "":
    print("（沒有 key，跳過 D 段）")
else:
    gemini = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
    print("已接上 Gemini：gemini-3.5-flash")
    print("注意：建立的寫法跟 A1 的 ChatOllama 幾乎一樣，只是換了 class")

### D3・同一個問題，問兩邊

同一句問題問兩邊——只是模型換了（gemma 取文字用 `.content`；新版 Gemini 回的是結構化內容，要用 `.text` 取乾淨文字）。

> 💰 **跑完盯著 token 用量看（實測數字，2026-07-16）：**
>
> | | input | output | 合計 |
> |---|---|---|---|
> | **gemma4（開源）** | 21 | **30** | 51 |
> | **gemini-3.5-flash（商用）** | 9 | **637**（reasoning 佔 601） | 646 |
>
> **兩邊的答案長度差不多、講得也差不多好——但商用旗艦光是「思考」就燒掉十幾倍 token（這次 output 是 gemma 的 20 倍、合計約 13 倍）。**
> 因為新一代商用模型會**先「想」再答**，那些看不見的「思考」token（`output_token_details.reasoning`）**照樣算錢**。
>
> **貴不一定值得，要看場景。** 每次跑數字會略有不同（reasoning token 尤其會變），但量級就是這樣。

In [ ]:
question = "用一句話解釋什麼是毛利率"

print("【開源模型 gemma4（Lab 2 一路用的）】")
resp_open = llm.invoke(question)
print(resp_open.content)
print("token 用量：", resp_open.usage_metadata)
print()

print("【商用模型 gemini-3.5-flash】")
if gemini is None:
    print("（沒有 key，跳過）")
else:
    resp_gemini = gemini.invoke(question)
    print(resp_gemini.text)   # 💡 新版 Gemini 回「內容區塊」而非純字串——用 .text 取乾淨文字（.content 會是一包結構）
    print("token 用量：", resp_gemini.usage_metadata)

### D4・把「它不可能知道的數字」也丟給商用旗艦

C 段你已經看過：問開源模型「宏圖飲料的 EPS」，它答不出來。**商用旗艦會怎麼回？**

> **實測（2026-07-15，跑 3 次都一樣）：Gemini 也老實說「找不到這家公司」**，還反問你是不是未上市、要不要提供股票代號。
> **兩邊都不會憑空捏造一家不存在的公司。**
>
> ⚠️ **但這不代表沒有幻覺。** 真正危險的幻覺不是「無中生有」，而是「**拿你給的數字往下推**」——
> 下午 Lab 3 你會親眼看到：**同一份資料、換個問法**，它就會算出資料裡根本沒有的數字。

In [ ]:
question = "宏圖飲料這家公司最近一季的 EPS 是多少？"

print("【開源模型 gemma4】")
print(llm.invoke(question).content)
print()

print("【商用模型 gemini-3.5-flash】")
if gemini is None:
    print("（沒有 key，跳過）")
else:
    print(gemini.invoke(question).text)

### 🔑 D 段收束：兩條路，各有各的適用場合

| | 開源模型（今天用的 gemma4） | 商用託管（Gemini / ChatGPT / Copilot） |
|---|---|---|
| **資料隱私** | 可以**自己架在公司機器上**，資料不出門 | 資料**送到供應商的雲**——金融法遵最卡的一關 |
| **成本** | 模型免費，但要自備算力與維運人力 | 按用量付費，起步幾乎零成本；量爆大時帳單可觀 |
| **能力上限** | 受開源模型天花板限制 | 用得到**最強旗艦** |
| **維運** | 要自己裝、調、顧機器 | 供應商全包，換模型只是改一個字串 |

> ⚠️ **一個誠實的說明**：今天為了省事，我們的 `gemma4` 其實是**跑在 Ollama 的雲上**，不是真的架在你的機器裡。
> 但**同一顆開源模型，你完全可以把它裝進公司的機器**——那時候「資料不出門」才真正成立。這就是「開源自架」的價值。

**下午區塊 3 會把這張表完整攤開**：什麼場景該自架、什麼場景該用商用（客戶個資 → 自架；行銷文案 → 商用）。

---

---
## 🛟 跟不上時的後路

| 狀況 | 怎麼辦 |
|---|---|
| `KeyError: 'OLLAMA_API_KEY'` | key 沒設，或設完沒重開 kernel → 回檢查 2 |
| `model not found` | 模型名漏了 cloud tag（要 `gemma4:cloud`） |
| `401` / 認證失敗 | key 貼錯（前後別留空格、別加引號）→ 回 README 第 3 步 |
| 跑很久沒反應 | 雲端要 5～15 秒才回，**不是當掉**；真的太久就重跑那格 |
| **雲端整個不通 / 額度用完** | **降本地生成**——跑下面那格 |

### 🛟 Backup：雲端不通時，改用本機模型（平常不用跑）

雲端掛掉也不用停課：把 `llm` 換成本機模型就好——**品質降，但「呼叫 LLM 生成」這件事照樣練得到**，後面 A2～C2 全部照跑。

> 需要先在終端機下 `ollama pull llama3.2:3b`（2GB，教室機通常已預裝）。

In [ ]:
# 平常不用跑這格。雲端不通時，把下面兩行的 # 拿掉再跑，後面所有格子就會改用本機模型。

# llm = ChatOllama(model="llama3.2:3b")   # 不帶 base_url、不帶 key ＝ 打你自己的電腦（localhost）
# print("已改用本機模型：", llm.model)

print("（Backup 格：目前不需要執行）")

---
## 🎓 收尾：今天你做出了什麼

1. **第一支會呼叫 LLM 生成的程式**：`ChatOllama` 指雲端 → `invoke` 送問題 → `.content` 取回答。
2. **看得見的成本**：`usage_metadata` 裡的 token 數，就是雲端跟你收錢的依據。
3. **一個旋鈕**：`num_predict`（最多生幾個 token 就收手）。
4. **親眼看到「幻覺」，也親手治好它一次**：資料一起送 → 它就照著資料答。

**下午的 Lab 3** 要解決的正是 C2 留下的那個問題——**資料太多，塞不進 prompt**。
我們會用上午 Lab 1 的檢索先挑出最相關的幾段，再餵給今天這支生成程式。**那就是 RAG。**